# <font color="steelblue">Patrones de demanda</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

> **Guión de trabajo para el estudiante.** Este cuaderno es el **enunciado** de un proyecto de **aprendizaje no supervisado** que debéis **desarrollar vosotros**: objetivos, fases, tareas obligatorias, andamiaje mínimo de código (marcas `# TODO`) y criterios de evaluación. **No es una solución.**

## <font color="steelblue">Objetivos del proyecto</font>

Cada producto es una **serie temporal**: sus ventas a lo largo de 52 semanas dibujan un **patrón de demanda** (plano, estacional, creciente, decreciente, con picos…). El objetivo es **agrupar los productos por la forma de su patrón** (*time series clustering*), no por cuánto venden. El reto central, propio de este proyecto, es trabajar con **series temporales**:

* **Normalizar cada serie** para capturar la **forma** y no el **nivel** (si no, agruparás por volumen de ventas).
* Elegir la **representación** (serie normalizada vs **características** extraídas) y la **distancia** (euclídea vs **DTW**, *Dynamic Time Warping*).
* **Agrupar** con métodos específicos de series (**TimeSeriesKMeans**, **k-Shape**) y compararlos.
* **Caracterizar** y **nombrar** los patrones prototípicos y traducirlos en **gestión de inventario/promociones**.
* Construir un **widget interactivo en Colab** que explore los patrones.

## <font color="steelblue">El conjunto de datos</font>

**811 productos** × **52 semanas** (UCI *Sales Transactions Weekly*; Tan & Lau, 2014). **Sin valores perdidos**; todo **entero ≥ 0**.

* **`Product_Code`** — código del producto. **Eliminar antes de agrupar.**
* **`W0`, `W1`, …, `W51`** — unidades compradas en cada semana del año. En conjunto, la **serie de demanda** del producto.

> **Idea clave:** dos productos pueden tener el **mismo patrón** (p. ej. pico en Navidad) aunque uno venda 10× más que el otro. Como queremos agrupar por **forma**, casi siempre habrá que **normalizar cada serie** antes de medir distancias.
> *(La versión UCI original incluye columnas `Normalized`/`MIN`/`MAX`; en la del curso solo están `W0..W51`.)*

## <font color="steelblue">Reglas del juego</font>

1. **Quita `Product_Code`** y trabaja con la matriz de 52 semanas.
2. **Normaliza por serie** (cada producto respecto a su propia media/desv o min–max) para agrupar por **forma**, no por **nivel**. Justifica la elección y compara con no normalizar.
3. **Combina enfoques:** representa las series de **dos formas** (serie normalizada cruda vs **características** extraídas) y compara los grupos resultantes.
4. **Usa al menos un método específico de series** (DTW / **k-Shape**) además de uno clásico, y compáralos.
5. **Elige el número de grupos con criterios** (silueta, codo, dendrograma), no a ojo.
6. **Caracteriza los patrones:** dibuja la **forma prototípica** de cada grupo y **nómbralo**; sin esto el proyecto no vale.
7. **Reproducibilidad:** fija `random_state` y comenta la **estabilidad**.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
url = "https://raw.githubusercontent.com/ia4legos/MachineLearning/main/data/Sales_Transactions_Weekly.csv"
sales = pd.read_csv(url)
print(f"Dimensiones: {sales.shape[0]:,} productos × {sales.shape[1]} columnas")
sales.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA temporal</font>

**Tareas a desarrollar**
1. **Estructura.** Confirmad las 52 columnas semanales `W0..W51` + `Product_Code`.
2. **Visualización de series.** Dibujad las series de **una muestra** de productos (varias en un mismo gráfico). Observad la diversidad de patrones (planos, con picos, estacionales…).
3. **Nivel vs forma (clave).** Comparad el **volumen total** por producto (suma/medias): veréis que los **niveles** varían enormemente. ¿Qué pasaría si agruparais sin normalizar?
4. **Estadísticos temporales.** Semana media de máximo, % de semanas con 0 ventas, variabilidad… para intuir tipos de patrón.
5. **Conclusión:** 3–4 hallazgos.

> **A responder:** ¿por qué, si no normalizas, el *clustering* separará "productos que venden mucho" de "productos que venden poco" en lugar de **patrones**?

# <font color="steelblue">Fase 2 — Normalización por serie (forma vs nivel)</font>

**Tareas a desarrollar**
1. Construid la matriz `X` de series (`W0..W51`, sin `Product_Code`).
2. **Normalizad cada serie por separado** para que importe la **forma**:
   * **z-normalización** por fila (resta su media, divide por su desv.), o
   * **min–max** por fila, o
   * la `TimeSeriesScalerMeanVariance` de **tslearn**.
3. **Comparad** el efecto: dibujad la misma muestra de series **antes** y **después** de normalizar.
4. (Opcional) Suavizado ligero (media móvil) para reducir ruido semanal.

> **A responder:** ¿qué información **pierdes** al normalizar (el nivel) y por qué aquí **interesa** perderla?

# <font color="steelblue">Fase 3 — Representación y reducción de dimensión</font>

**Tareas a desarrollar**
1. **Representación A — serie normalizada cruda** (52 valores por producto).
2. **Representación B — características extraídas:** resumid cada serie con variables interpretables (p. ej. **pendiente/tendencia**, **fuerza estacional**, **nº de picos**, **autocorrelación lag-1**, **coef. de variación**, semana del máximo, % de ceros). Esto convierte el problema en un *clustering* tabular clásico.
3. **Visualización.** Aplicad **PCA** y/o **MDS**/**t-SNE** sobre la representación A para ver si hay estructura.
4. Comentad qué representación parece capturar mejor los patrones.

> **Combinar modelos:** en la Fase 4 agruparéis con **ambas** representaciones y compararéis.

# <font color="steelblue">Fase 4 — Clustering de series y combinación de modelos</font>

**Tareas a desarrollar**
1. **Método específico de series:** **`TimeSeriesKMeans`** de tslearn con distancia **euclídea** y con **DTW** (*Dynamic Time Warping*), y/o **`KShape`** (basado en correlación de formas).
2. **Método clásico sobre características:** K-Means / jerárquico sobre la **representación B**.
3. **Combina/compara:** ¿agrupan igual la serie normalizada (euclídeo/DTW/k-Shape) y las características? Mide la **concordancia** (Adjusted Rand Index).
4. Dibuja, para una solución, los **centroides** (formas medias) de cada grupo.

> **A responder:** ¿qué aporta **DTW** frente a la distancia euclídea cuando dos series tienen el mismo patrón **desfasado** en el tiempo?

# <font color="steelblue">Fase 5 — Validación y elección del número de grupos</font>

**Tareas a desarrollar**
1. **Número de grupos:** **codo** (inercia) y **silueta** (con distancia euclídea y, si puedes, **DTW**) variando `k`; **dendrograma** para el jerárquico.
2. **Índices internos** y comparación entre métodos/representaciones.
3. **Estabilidad** ante semillas/submuestras.
4. **Decisión justificada** del método, la representación y el número de patrones.

> El mejor `k` equilibra **calidad** (índices) e **interpretabilidad** (que los patrones sean distinguibles y con sentido de negocio).

# <font color="steelblue">Fase 6 — Caracterización de los patrones</font>

**Tareas a desarrollar**
1. **Forma prototípica.** Para cada grupo, dibuja la **serie media** (centroide) y una **banda** (± desv.): ese es el **patrón** del grupo.
2. **Etiqueta cada patrón:** plano/estable, **estacional**, creciente, decreciente, **con pico puntual**, intermitente (muchos ceros)…
3. **Tamaño y ejemplos.** Nº de productos por patrón y algunos productos representativos.
4. **Lectura de negocio.** ¿Qué implica cada patrón para **inventario**, **previsión** y **promociones** (p. ej. los estacionales requieren *stock* anticipado; los intermitentes, otra política)?

> Es la fase con **más peso**: el valor está en **nombrar y explicar** los patrones.

# <font color="steelblue">Fase 7 — Aplicación interactiva (widget en Colab)</font>

Aplicación **dentro del cuaderno** con **`ipywidgets`** (no despliegue). Implementa **al menos una**:

1. **Explorador de patrones:** un desplegable de grupo que muestre su **forma prototípica**, su tamaño y varios productos de ejemplo.
2. **Buscador por producto:** elige un `Product_Code` y muestra **su** serie, el **patrón** al que pertenece y la **recomendación** de gestión asociada a ese patrón.
3. **(Opcional)** Dispersión PCA/MDS coloreada por patrón con `plotly`.

> El widget solo **usa** el modelo ya ajustado; no reentrena en cada interacción.

# <font color="steelblue">Entregables y formato</font>

* **Cuaderno** ejecutable de principio a fin, con código y **celdas de texto** que justifiquen cada decisión (normalización, distancia, nº de patrones, interpretación).
* **Memoria breve:** EDA temporal, normalización, representaciones y reducción, comparación de métodos de *clustering* de series, validación, **caracterización y nombres de los patrones**, y la aplicación.
* **Widget(s)** funcionando dentro del cuaderno.
* **Reproducibilidad:** `random_state` fijado y comentario de estabilidad.

## <font color="steelblue">Rúbrica de evaluación (100 pts)</font>

| Fase | Qué se valora | Puntos |
|---|---|---:|
| 1. EDA temporal | Visualización de series, **nivel vs forma**, estadísticos | 12 |
| 2. Normalización por serie | **Forma vs nivel**, comparación antes/después, justificación | 16 |
| 3. Representación y reducción | Serie normalizada vs **características**, PCA/MDS | 14 |
| 4. Clustering de series | Euclídeo + **DTW/k-Shape** + por características, comparación | 18 |
| 5. Validación y nº de grupos | Codo/silueta(DTW)/dendrograma, índices, **estabilidad** | 12 |
| 6. Caracterización de patrones | **Formas prototipo**, nombres, lectura de inventario | 18 |
| 7. Aplicación (widget) | `ipywidgets` funcional (explorador / buscador) | 10 |
| — Extra | *DTW barycenter*, k-Shape, características con `tsfresh`, euclídeo vs DTW, mapa `plotly` | +10 |

> **Penalizaciones:** **agrupar sin normalizar** (acabar segmentando por volumen), elegir el nº de grupos **a ojo**, no dibujar/nombrar las **formas prototipo**, o ignorar que los datos son **series temporales**.

# <font color="steelblue">Pistas y errores típicos</font>

* **Normaliza por serie.** Sin ello agruparás "best-sellers vs nicho", no patrones de demanda.
* **DTW para desfases.** Si dos series tienen el mismo patrón desplazado unas semanas, la euclídea las separa y **DTW** las junta.
* **Dos representaciones.** La serie cruda (con DTW/k-Shape) capta la forma; las **características** (tendencia, estacionalidad, picos) dan interpretabilidad. Compáralas.
* **Caracterizar = dibujar la forma.** El centroide de cada grupo **es** el patrón: nómbralo (estacional, plano, pico…).
* **Negocio:** cada patrón implica una política de **inventario/promoción** distinta.
* **Widget, no despliegue:** `ipywidgets` dentro de Colab.

# <font color="steelblue">Referencias</font>

* Tan, S. C. & Lau, J. P. S. (2014). *Time series clustering: A superior alternative for market basket analysis*. DaEng-2013, Springer.
* *Sales Transactions Dataset Weekly*. UCI ML Repository (id 396).
* Tavenard, R. et al. *tslearn: A Machine Learning Toolkit for Time Series Data*.
* Cuadernos del curso: *Componentes principales y variantes*, *Escalas multidimensionales (MDS)*, *Clustering (K-Means, jerárquico, DBSCAN, mixturas gaussianas)*.